In [1]:
!pwd

/home/user/Documents


In [3]:
import tensorflow
print(tensorflow.__version__)

2.19.1


In [2]:
import glob
import os
import re
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.utils import to_categorical

In [3]:
import sys  
sys.path.insert(0, '../home/user/Documents')

In [4]:
from dotenv import load_dotenv
import os

load_dotenv("/home/user/.env")
DagsHub_username = os.getenv("DagsHub_username")
DagsHub_token=os.getenv("DagsHub_token")

print(os.getenv("DagsHub_username"))
print(os.getenv("DagsHub_token"))

emnaaS
ac026871943a805c04cfaa7af4cf1a0734c83779


In [16]:
from kmeans_data import load_image_dataset
images, labels = load_image_dataset(
    image_dir="/home/user/datasets/splits/batch_2500",
    labels_file="/home/user/datasets/trainLabels_cropped.csv",
    max_images=2500)

Loaded 2500 images with labels


In [17]:
from kmeans_data import prepare_data
images_flat, labels_flat = prepare_data(images, labels)

images shape (2500, 196608)
labels shape (2500,)


In [8]:
#test d'un autre code
from sequential_kmeans import cust_K_Means
import os
import time
import json
import numpy as np
import mlflow
import time
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import silhouette_score
from sklearn.metrics import davies_bouldin_score
seq_kmeans = cust_K_Means(3, 300)
seq_kmeans.fit(images_flat)
y_predict = seq_kmeans.labels_
# Get predictions (cluster labels)
dbi_score = davies_bouldin_score(images_flat, y_predict)
ch_score = calinski_harabasz_score(images_flat, y_predict)
score = silhouette_score(images_flat, y_predict)
print(f"Silhouette Score: {score:.4f}")
print(f"Calinski-Harabasz Index: {ch_score:.4f}")
print(f"Davies-Bouldin Index: {dbi_score:.4f}")


Converged after 23 iterations
Silhouette Score: 0.1995
Calinski-Harabasz Index: 619.4676
Davies-Bouldin Index: 1.5305


In [11]:
import os
os.environ['MLFLOW_TRACKING_USERNAME']= DagsHub_username
os.environ["MLFLOW_TRACKING_PASSWORD"] = DagsHub_token

In [12]:
import dagshub
dagshub.init(repo_owner='emnaaS', repo_name='kmeans_repo', mlflow=True)

Accessing as emnaaS

Initialized MLflow to track repo "emnaaS/kmeans_repo"

Repository emnaaS/kmeans_repo initialized!

In [13]:
import mlflow
mlflow.set_experiment("4ème_batch")

<Experiment: artifact_location='mlflow-artifacts:/4c2ef0f1ff074462bd0701a6a714aaf6', creation_time=1764067801212, experiment_id='5', last_update_time=1764067801212, lifecycle_stage='active', name='4ème_batch', tags={}>

In [22]:
from sequential_kmeans import cust_K_Means
import os
import time
import json
import numpy as np
import mlflow
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import silhouette_score
from sklearn.metrics import davies_bouldin_score
import time

# Start MLflow run
with mlflow.start_run(run_name="seq"):

    # MLflow: log parameters
    mlflow.log_param("n_clusters", 3)
    mlflow.log_param("max_iter", 300)
    mlflow.log_param("num_workers", 1)
    mlflow.log_param("num_folds", 10)
    mlflow.log_param("num_images", len(images_flat))
    
    num_features = images_flat.shape[1]
    mlflow.log_param("num_features", num_features)
    tr_time = []
    for i in range(10):
        print(f"-----------------------------------------------------------------{i+1}---------------------------------------------------------------")
        seq_kmeans = cust_K_Means(3, 300)
        debut = time.time()
        seq_kmeans.fit(images_flat)
        fin = time.time()
        tr_time.append(fin-debut)
    training_time = np.mean(tr_time)
    mlflow.log_metric("avg training time", training_time)
    num_iterations = seq_kmeans.number_of_iter
    mlflow.log_metric("iterations", num_iterations)
    cluster_labels = seq_kmeans.labels_
    # Compute cluster sizes
    unique, counts = np.unique(cluster_labels, return_counts=True)
    cluster_size_dict = {int(k): int(v) for k, v in zip(unique, counts)}
    # MLflow: Log size of each cluster
    for cluster_id, size in cluster_size_dict.items():
        mlflow.log_metric(f"cluster_{cluster_id}_size", size)

    y_pred = seq_kmeans.labels_
    # Get predictions (cluster labels)
    dbi_score = davies_bouldin_score(images_flat, y_pred)
    mlflow.log_metric("davies bouldin score ", dbi_score)
    ch_score = calinski_harabasz_score(images_flat, y_pred)
    mlflow.log_metric("calinski harabasz score ", ch_score)
    score = silhouette_score(images_flat, y_pred)
    mlflow.log_metric("silhouette score ", score)
    print("\n✓ Logged centroids, iterations, and cluster sizes to MLflow successfully.\n")
        

-----------------------------------------------------------------1---------------------------------------------------------------
Converged after 16 iterations
-----------------------------------------------------------------2---------------------------------------------------------------
Converged after 16 iterations
-----------------------------------------------------------------3---------------------------------------------------------------
Converged after 16 iterations
-----------------------------------------------------------------4---------------------------------------------------------------
Converged after 16 iterations
-----------------------------------------------------------------5---------------------------------------------------------------
Converged after 16 iterations
-----------------------------------------------------------------6---------------------------------------------------------------
Converged after 16 iterations
----------------------------------------

In [17]:
from sklearn.metrics import davies_bouldin_score
y_pred = seq_kmeans.labels_
# Get predictions (cluster labels)
dbi_score = davies_bouldin_score(images_flat, y_pred)
print(f"Davies-Bouldin Index: {dbi_score:.4f}")

Davies-Bouldin Index: 1.5304


In [18]:
from sklearn.metrics import davies_bouldin_score
y_predict = kmeans.labels_
# scikit learn
dbi_score = davies_bouldin_score(images_flat, y_predict)
print(f"Davies-Bouldin Index: {dbi_score:.4f}")

Davies-Bouldin Index: 1.5390


In [19]:
from sklearn.metrics import calinski_harabasz_score
ch_score = calinski_harabasz_score(images_flat, y_pred)
print(f"Calinski-Harabasz Index: {ch_score:.4f}")

Calinski-Harabasz Index: 619.4527


In [20]:
from sklearn.metrics import calinski_harabasz_score
ch_score = calinski_harabasz_score(images_flat, y_predict)
print(f"Calinski-Harabasz Index: {ch_score:.4f}")

Calinski-Harabasz Index: 619.5422


In [21]:
from sklearn.metrics import silhouette_score

# Compute silhouette score
score = silhouette_score(images_flat, y_pred)
print(f"Silhouette Score: {score:.4f}")

Silhouette Score: 0.2007


In [22]:
from sklearn.metrics import silhouette_score

# Compute silhouette score
score = silhouette_score(images_flat, y_predict)
print(f"Silhouette Score: {score:.4f}")

Silhouette Score: 0.1991


In [16]:
from sklearn.cluster import KMeans

# X is your flattened image matrix: shape (n_samples, n_features)
k = 10  # number of clusters you want
with mlflow.start_run(run_name="scikit_learn"):

    # MLflow: log parameters
    mlflow.log_param("n_clusters", 3)
    mlflow.log_param("max_iter", 300)
    mlflow.log_param("num_workers", 1)
    mlflow.log_param("num_folds", 1)
    mlflow.log_param("num_images", len(images_flat))
    
    num_features = images_flat.shape[1]
    mlflow.log_param("num_features", num_features)
    kmeans = KMeans(n_clusters=3, max_iter=300)
    kmeans.fit(images_flat)
    
    labels = kmeans.labels_
    centroids = kmeans.cluster_centers_
    unique, counts = np.unique(labels, return_counts=True)
    cluster_size_dict = {int(k): int(v) for k, v in zip(unique, counts)}
    # MLflow: Log size of each cluster
    for cluster_id, size in cluster_size_dict.items():
        mlflow.log_metric(f"cluster_{cluster_id}_size", size)
    print("\n✓ Logged centroids, iterations, and cluster sizes to MLflow successfully.\n")



✓ Logged centroids, iterations, and cluster sizes to MLflow successfully.

🏃 View run scikit_learn at: https://dagshub.com/emnaaS/kmeans_repo.mlflow/#/experiments/4/runs/dafb4c42f0b643d1bd0c255a44dda2f7
🧪 View experiment at: https://dagshub.com/emnaaS/kmeans_repo.mlflow/#/experiments/4


In [50]:
print(kmeans.tol)


0.0001


In [9]:
!ls ~/Documents

 kmeans_dask.py		     kmeans_ray.py   runinfo
 kmeans_data.py		     mémoire	     sequential_kmeans.py
 kmeans_multiprocessing.py   modeling.py    'testing modularisation.ipynb'
 kmeans_parsl.py	     __pycache__     Untitled.ipynb


In [18]:
from kmeans_ray import K_Means_Distributed
import os
import time
import json
import numpy as np
import mlflow
import ray
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import silhouette_score
from sklearn.metrics import davies_bouldin_score

# Start MLflow run
with mlflow.start_run(run_name="ray2"):

    # MLflow: log parameters
    mlflow.log_param("n_clusters", 3)
    mlflow.log_param("max_iter", 300)
    mlflow.log_param("num_workers", 2)
    mlflow.log_param("num_folds", 10)
    mlflow.log_param("num_images", len(images_flat))
    
    # Ray memory configuration
    os.environ['RAY_memory_usage_threshold'] = '0.97'
    os.environ['RAY_memory_monitor_refresh_ms'] = '250'
    num_features = images_flat.shape[1]
    mlflow.log_param("num_features", num_features)

    # Shutdown previous Ray cluster if needed
    if ray.is_initialized():
        ray.shutdown()

    # Init Ray
    ray.init(num_cpus=2)
    print(ray.cluster_resources())
    ray_time =[]
    for i in range(10):
        # Train model
        kmeans_dist = K_Means_Distributed(
            n_clusters=3,
            max_iter=300,
            num_workers=2
        )
        start_time = time.time()
        kmeans_dist.fit(images_flat)
        end_time = time.time()
        ray_time.append(end_time-start_time)
    
        training_time = np.mean(ray_time)
        print(f"Training time = {training_time:.2f} seconds")

    # MLflow: log training time
    mlflow.log_metric("avg training time", training_time)

    # Retrieve results from your KMeans class
    centroids = kmeans_dist.cluster_centers_
    num_iterations = kmeans_dist.number_of_iter
    cluster_labels = kmeans_dist.labels_

    # MLflow: Log iteration count
    mlflow.log_metric("iterations", num_iterations)

    # Compute cluster sizes
    unique, counts = np.unique(cluster_labels, return_counts=True)
    cluster_size_dict = {int(k): int(v) for k, v in zip(unique, counts)}

    # MLflow: Log size of each cluster
    for cluster_id, size in cluster_size_dict.items():
        mlflow.log_metric(f"cluster_{cluster_id}_size", size)

    y_pred = kmeans_dist.labels_
    # Get predictions (cluster labels)
    dbi_score = davies_bouldin_score(images_flat, y_pred)
    mlflow.log_metric("davies bouldin score ", dbi_score)
    ch_score = calinski_harabasz_score(images_flat, y_pred)
    mlflow.log_metric("calinski harabasz score ", ch_score)
    score = silhouette_score(images_flat, y_pred)
    mlflow.log_metric("silhouette score ", score)
    print("\n✓ Logged centroids, iterations, and cluster sizes to MLflow successfully.\n")


/home/user/anaconda3/envs/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-11-29 10:29:07,725	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2025-11-29 10:29:16,874	INFO worker.py:2013 -- Started a local Ray instance.
/home/user/anaconda3/envs/myenv/lib/python3.12/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


{'object_store_memory': 8798601216.0, 'CPU': 2.0, 'node:__internal_head__': 1.0, 'node:192.168.62.206': 1.0, 'memory': 20530069504.0}
Data splitting & Ray.put: 1.012s

=== Training Started ===

--- Iteration 1 ---
Ray.get (collection): 8.126s
Result aggregation: 0.013s
(Worker pid=710861) Worker - Distance computation: 6.401s | Aggregation: 0.713s | Total: 8.013s
Centroid update: 0.008s
Convergence check: 0.006s | Center shift: 284.277995

--- Iteration 2 ---
(Worker pid=710860) Worker - Distance computation: 6.187s | Aggregation: 0.782s | Total: 7.575s [repeated 2x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
Ray.get (collection): 7.880s
Result aggregation: 0.018s
Centroid update: 0.004s
Convergence check: 0.004s | Center shift: 93.995260

--- Iteration 3 ---
(Worker pid=710860) Worker - Distance computa

In [17]:
ray.shutdown()

In [ ]:
from sklearn.metrics import davies_bouldin_score
y_ray = kmeans_dist.labels_
# scikit learn
dbi_score = davies_bouldin_score(images_flat, y_ray)
print(f"Davies-Bouldin Index: {dbi_score:.4f}")
from sklearn.metrics import calinski_harabasz_score
ch_score = calinski_harabasz_score(images_flat, y_ray)
print(f"Calinski-Harabasz Index: {ch_score:.4f}")
from sklearn.metrics import silhouette_score

# Compute silhouette score
score = silhouette_score(images_flat, y_ray)
print(f"Silhouette Score: {score:.4f}")

In [18]:
len(ray_time)

10

In [23]:
ray.shutdown()

In [66]:
import shutil
from IPython.display import FileLink

# Replace 'your_folder_name' with your actual folder path
folder_name = '/home/user/datasets/splits'

# Create ZIP archive
shutil.make_archive('splits', 'zip', folder_name)

# Generate download link
FileLink('splits.zip')

/home/user/Documents/splits.zip

In [24]:
print(images_flat.dtype)
kmeans_parsl.dtype

float32


AttributeError: 'KMeansDistributedParsl' object has no attribute 'dtype'

In [16]:
parsl.clear()

In [10]:
from kmeans_parsl import KMeansDistributedParsl
import parsl
from parsl.app.app import python_app
from parsl.config import Config
from parsl.executors import ThreadPoolExecutor
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import silhouette_score
from sklearn.metrics import davies_bouldin_score

# Start MLflow run
with mlflow.start_run(run_name="parsl12"):
    # Setup Parsl configuration
    # Disable all parsl logging before loading
    import logging
    # MLflow: log parameters
    mlflow.log_param("n_clusters", 3)
    mlflow.log_param("max_iter", 300)
    mlflow.log_param("num_workers", 12)
    mlflow.log_param("num_folds", 10)
    mlflow.log_param("num_images", len(images_flat))
    num_features = images_flat.shape[1]
    mlflow.log_param("num_features", num_features)
    
    parsl.set_stream_logger(level=logging.CRITICAL)
    config = Config(
            executors=[
                    ThreadPoolExecutor(max_threads=12, label='local_threads')
             ],
        monitoring=None,  # Disable monitoring
        usage_tracking=False
        )
    parsl.load(config)
    parsl_time=[]
    for i in range(10):
        print(f'---------------------------------------{i+1}------------------------------')
        kmeans_parsl = KMeansDistributedParsl(n_clusters=3, n_workers=12)
        # Create and fit model
        import time
        debut = time.time()
        kmeans_parsl.fit(images_flat)
        fin = time.time()
        parsl_time.append(fin-debut)
    
    
    training_time = np.mean(parsl_time)
    print(f"Training time = {training_time:.2f} seconds")

    # MLflow: log training time
    mlflow.log_metric("avg training time", training_time)

    # Retrieve results from your KMeans class
    centroids = kmeans_parsl.cluster_centers_
    num_iterations = kmeans_parsl.number_of_iter
    cluster_labels = kmeans_parsl.labels_

    # MLflow: Log iteration count
    mlflow.log_metric("iterations", num_iterations)

    # Compute cluster sizes
    unique, counts = np.unique(cluster_labels, return_counts=True)
    cluster_size_dict = {int(k): int(v) for k, v in zip(unique, counts)}

    # MLflow: Log size of each cluster
    for cluster_id, size in cluster_size_dict.items():
        mlflow.log_metric(f"cluster_{cluster_id}_size", size)

    y_pred = kmeans_parsl.labels_
    # Get predictions (cluster labels)
    dbi_score = davies_bouldin_score(images_flat, y_pred)
    mlflow.log_metric("davies bouldin score ", dbi_score)
    ch_score = calinski_harabasz_score(images_flat, y_pred)
    mlflow.log_metric("calinski harabasz score ", ch_score)
    score = silhouette_score(images_flat, y_pred)
    mlflow.log_metric("silhouette score ", score)


    print("\n✓ Logged centroids, iterations, and cluster sizes to MLflow successfully.\n")

    
    # Clean up
    parsl.clear()

---------------------------------------1------------------------------

=== Setup ===
Data splitting: 0.000s

=== Training Started ===

--- Iteration 1 ---
Submit tasks: 0.049s
Gather results (parallel work): 3.196s
Result aggregation: 0.083s
Centroid update: 0.013s
Convergence check: 0.008s | Center shift: 323.257310
Iteration overhead (print, misc): 0.001s
Total iteration time: 3.349s

--- Iteration 2 ---
Submit tasks: 0.045s
Gather results (parallel work): 4.265s
Result aggregation: 0.024s
Centroid update: 0.005s
Convergence check: 0.006s | Center shift: 77.055547
Iteration overhead (print, misc): 0.001s
Total iteration time: 4.345s

--- Iteration 3 ---
Submit tasks: 0.031s
Gather results (parallel work): 4.284s
Result aggregation: 0.024s
Centroid update: 0.005s
Convergence check: 0.004s | Center shift: 18.500586
Iteration overhead (print, misc): 0.001s
Total iteration time: 4.349s

--- Iteration 4 ---
Submit tasks: 0.050s
Gather results (parallel work): 4.120s
Result aggregation: 0

In [16]:
from kmeans_dask import KMeansDistributedDask
import numpy as np
from dask.distributed import Client
import time
import mlflow
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import silhouette_score
from sklearn.metrics import davies_bouldin_score


# Start MLflow run
with mlflow.start_run(run_name="dask12"):
    # Setup Parsl configuration
    # Disable all parsl logging before loading
    import logging
    # MLflow: log parameters
    mlflow.log_param("n_clusters", 3)
    mlflow.log_param("max_iter", 300)
    mlflow.log_param("num_workers", 12)
    mlflow.log_param("num_folds", 10)
    mlflow.log_param("num_images", len(images_flat))
    num_features = images_flat.shape[1]
    mlflow.log_param("num_features", num_features)

    
    print("Starting Dask Client...")
    client = Client(n_workers=1, threads_per_worker=12)
    dask_time=[]
    for i in range(10):
        print(f'-------------------------------------{i+1}-----------------------------')
        kmeans_dask = KMeansDistributedDask(n_clusters=3, max_iter=300, n_workers=12)
        start = time.time()
        kmeans_dask.fit(images_flat, client)
        end = time.time()
        dask_time.append(end-start)

    training_time = np.mean(dask_time)
    print(f"Training time = {training_time:.2f} seconds")

    # MLflow: log training time
    mlflow.log_metric("avg training time", training_time)

    # Retrieve results from your KMeans class
    centroids = kmeans_dask.cluster_centers_
    num_iterations = kmeans_dask.number_of_iter
    cluster_labels = kmeans_dask.labels_

    # MLflow: Log iteration count
    mlflow.log_metric("iterations", num_iterations)

    # Compute cluster sizes
    unique, counts = np.unique(cluster_labels, return_counts=True)
    cluster_size_dict = {int(k): int(v) for k, v in zip(unique, counts)}

    # MLflow: Log size of each cluster
    for cluster_id, size in cluster_size_dict.items():
        mlflow.log_metric(f"cluster_{cluster_id}_size", size)

    y_pred = kmeans_dask.labels_
    # Get predictions (cluster labels)
    dbi_score = davies_bouldin_score(images_flat, y_pred)
    mlflow.log_metric("davies bouldin score ", dbi_score)
    ch_score = calinski_harabasz_score(images_flat, y_pred)
    mlflow.log_metric("calinski harabasz score ", ch_score)
    score = silhouette_score(images_flat, y_pred)
    mlflow.log_metric("silhouette score ", score)

    

    print("\n✓ Logged centroids, iterations, and cluster sizes to MLflow successfully.\n")
    client.close()



Starting Dask Client...
-------------------------------------1-----------------------------

=== Setup ===
Data splitting: 0.000s
Scatter to workers: 9.027s

=== Training Started ===

--- Iteration 1 ---
Submit tasks: 0.161s
Gather results: 4.079s
Result aggregation: 0.013s
Centroid update: 0.002s
Convergence check: 0.002s | Center shift: 323.257310

--- Iteration 2 ---
Submit tasks: 0.168s
Gather results: 4.471s
Result aggregation: 0.013s
Centroid update: 0.003s
Convergence check: 0.003s | Center shift: 77.055547

--- Iteration 3 ---
Submit tasks: 0.181s
Gather results: 4.550s
Result aggregation: 0.014s
Centroid update: 0.004s
Convergence check: 0.002s | Center shift: 18.500586

--- Iteration 4 ---
Submit tasks: 0.170s
Gather results: 4.369s
Result aggregation: 0.013s
Centroid update: 0.003s
Convergence check: 0.003s | Center shift: 5.501034

--- Iteration 5 ---
Submit tasks: 0.164s
Gather results: 4.312s
Result aggregation: 0.013s
Centroid update: 0.003s
Convergence check: 0.002s | C

In [15]:
from kmeans_multiprocessing import KMeansMultiprocessing
import mlflow.pyfunc
import mlflow
import numpy as np
from multiprocessing import Pool, cpu_count, shared_memory
import os
import ctypes
import time
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import silhouette_score
from sklearn.metrics import davies_bouldin_score

# Start MLflow run
with mlflow.start_run(run_name="mult12"):
    # Setup Parsl configuration
    # Disable all parsl logging before loading
    import logging
    # MLflow: log parameters
    mlflow.log_param("n_clusters", 3)
    mlflow.log_param("max_iter", 300)
    mlflow.log_param("num_workers", 12)
    mlflow.log_param("num_folds", 10)
    mlflow.log_param("num_images", len(images_flat))
    num_features = images_flat.shape[1]
    mlflow.log_param("num_features", num_features)
    mult_time =[]
    for i in range(10):
        print(f"------------------------------------------------------------------{i+1}------------------------------------------------------------")
        debut = time.time()
        kmeans_mult = KMeansMultiprocessing(n_clusters= 3, max_iter=300, n_workers=12, use_shared_memory=True)
        kmeans_mult.fit(images_flat)
        fin = time.time()
        mult_time.append(fin-debut)

    training_time = np.mean(mult_time)
    print(f"Training time = {training_time:.2f} seconds")

    # MLflow: log training time
    mlflow.log_metric("avg training time", training_time)

    # Retrieve results from your KMeans class
    centroids = kmeans_mult.cluster_centers_
    num_iterations = kmeans_mult.number_of_iter
    cluster_labels = kmeans_mult.labels_

    # MLflow: Log iteration count
    mlflow.log_metric("iterations", num_iterations)

    # Compute cluster sizes
    unique, counts = np.unique(cluster_labels, return_counts=True)
    cluster_size_dict = {int(k): int(v) for k, v in zip(unique, counts)}

    # MLflow: Log size of each cluster
    for cluster_id, size in cluster_size_dict.items():
        mlflow.log_metric(f"cluster_{cluster_id}_size", size)

    y_pred = kmeans_mult.labels_
    # Get predictions (cluster labels)
    dbi_score = davies_bouldin_score(images_flat, y_pred)
    mlflow.log_metric("davies bouldin score ", dbi_score)
    ch_score = calinski_harabasz_score(images_flat, y_pred)
    mlflow.log_metric("calinski harabasz score ", ch_score)
    score = silhouette_score(images_flat, y_pred)
    mlflow.log_metric("silhouette score ", score)
    

    print("\n✓ Logged centroids, iterations, and cluster sizes to MLflow successfully.\n")
    


------------------------------------------------------------------1------------------------------------------------------------

=== Setup ===
Shared memory creation and data copy: 2.939s
Chunk boundary calculation: 0.001s

=== Training Started ===

--- Iteration 1 ---
Arguments preparation: 0.000s
Parallel computation (pool.map): 2.861s
Result aggregation: 0.016s
Centroid update: 0.005s
Convergence check: 0.006s | Center shift: 323.257310
Iteration overhead (print, misc): 0.001s
Total iteration time: 2.890s

--- Iteration 2 ---
Arguments preparation: 0.000s
Parallel computation (pool.map): 3.617s
Result aggregation: 0.022s
Centroid update: 0.003s
Convergence check: 0.003s | Center shift: 77.055547
Iteration overhead (print, misc): 0.001s
Total iteration time: 3.646s

--- Iteration 3 ---
Arguments preparation: 0.000s
Parallel computation (pool.map): 3.905s
Result aggregation: 0.013s
Centroid update: 0.003s
Convergence check: 0.003s | Center shift: 18.500586
Iteration overhead (print, m

In [36]:
!python --version

Python 3.13.5
